# GeoNames-first local LLM and vector resolution

This experiment keeps GeoNames as the authority. A local LLM extracts search terms from difficult locality text, PostgreSQL retrieves bounded GeoNames candidates, local embeddings rank those candidates, and the LLM may choose only one of the supplied GeoNames IDs.

No locality text leaves the machine. The model is not allowed to invent coordinates or identifiers.

In [ ]:
from __future__ import annotations

import json
import re
from functools import lru_cache

import numpy as np
import pandas as pd
import psycopg2
import requests
from tqdm.auto import tqdm

from pkb.config import settings
from pkb.models.places import search_geonames_candidates
from pkb.utils import country_string_to_iso


In [ ]:
PLACEBOT_COMMIT = "b036dd0b555d5ff32f7775bc0461707cc0df6285"
PLACEBOT_RAW = (
    "https://raw.githubusercontent.com/JackDanHollister/PlaceBot/"
    f"{PLACEBOT_COMMIT}/benchmarks/data"
)
SAMPLE_PER_DATASET = 50

benchmark_files = {
    "bombus_uk": f"{PLACEBOT_RAW}/bombus_uk_reference.tsv",
    "odonata_global": f"{PLACEBOT_RAW}/odonata_global_reference.tsv",
}

benchmarks = pd.concat(
    [
        pd.read_csv(url, sep="\t").head(SAMPLE_PER_DATASET).assign(benchmark=name)
        for name, url in benchmark_files.items()
    ],
    ignore_index=True,
)

benchmarks[["benchmark", "Barcode", "label_verbatim", "Country"]].head()


In [ ]:
OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "llama3.1:latest"


def ollama_json(prompt, max_tokens=120):
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
                "seed": 0,
                "num_predict": max_tokens,
            },
        },
        timeout=180,
    )
    response.raise_for_status()
    return json.loads(response.json()["response"])


def embed(texts):
    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": OLLAMA_MODEL, "input": texts},
        timeout=180,
    )
    response.raise_for_status()
    vectors = np.asarray(response.json()["embeddings"], dtype=float)
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


## Parse, retrieve and rank

Simple strings use deterministic delimiter splitting. Complex strings use the local model to return at most five search terms. Every returned term is validated before it reaches PostgreSQL.

Vectors rerank the bounded lexical candidate set; they do not replace country filtering or GeoNames identity retrieval.

In [ ]:
DISTANCE_PREFIX = re.compile(
    r"^\s*\d+(?:\.\d+)?\s*(?:km|kms?|m|mi|miles?)\.?\s+"
    r"(?:north|south|east|west|n|s|e|w|ne|nw|se|sw)\.?\s+(?:of\s+)?",
    re.IGNORECASE,
)
LABEL_PREFIX = re.compile(r"^\s*\d+(?:st|nd|rd|th)\s+label\s*:\s*", re.IGNORECASE)


def simple_components(locality, country):
    country = str(country or "").strip().casefold()
    components = []
    for value in re.split(r"[,;:]", str(locality or "")):
        value = LABEL_PREFIX.sub("", value).strip(" .")
        value = DISTANCE_PREFIX.sub("", value).strip(" .")
        if len(value) >= 3 and value.casefold() != country:
            components.append(value)
    return list(dict.fromkeys(components))


@lru_cache(maxsize=None)
def search_terms(locality, country):
    components = simple_components(locality, country)
    if len(components) <= 1:
        return components

    parsed = ollama_json(
        "Extract up to five place-name search terms from this museum locality. "
        "Expand obvious geographic abbreviations, but do not invent places. "
        "Order terms from most specific to broadest. Return JSON with only a "
        f"'search_terms' array.\nLocality: {locality}\nCountry: {country}"
    )
    terms = [
        str(term).strip()
        for term in parsed.get("search_terms", [])
        if len(str(term).strip()) >= 3
    ]
    return list(dict.fromkeys(terms))[:5] or components


In [ ]:
connection = psycopg2.connect(
    host=settings.postgres_host,
    port=settings.postgres_port,
    dbname=settings.postgres_database,
    user=settings.postgres_user,
    password=settings.postgres_password,
)


@lru_cache(maxsize=None)
def search_component(component, country_code):
    return search_geonames_candidates(
        connection,
        component,
        country_code=country_code,
        limit=5,
    )


def candidate_pool(row):
    country_code = country_string_to_iso(str(row.Country))
    candidates = {}
    for component in search_terms(str(row.label_verbatim), str(row.Country)):
        for candidate in search_component(component, country_code):
            candidate = {**candidate, "query_component": component}
            current = candidates.get(candidate["geoname_id"])
            if current is None or (
                candidate["exact_match"], candidate["name_score"]
            ) > (current["exact_match"], current["name_score"]):
                candidates[candidate["geoname_id"]] = candidate
    return list(candidates.values())


In [ ]:
def candidate_text(candidate):
    return (
        f"{candidate['canonical_name']}; matched name {candidate['matched_name']}; "
        f"query term {candidate['query_component']}; country {candidate['country_code']}; "
        f"feature {candidate['feature_code']}"
    )


def vector_rank(locality, country, candidates):
    if not candidates:
        return []
    texts = [f"{locality}; country {country}"] + [
        candidate_text(candidate) for candidate in candidates
    ]
    vectors = embed(texts)
    scores = vectors[1:] @ vectors[0]
    return [
        {**candidate, "vector_score": float(score)}
        for candidate, score in sorted(
            zip(candidates, scores),
            key=lambda value: value[1],
            reverse=True,
        )
    ]


def llm_choice(locality, country, candidates):
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0]

    options = [
        {
            "geoname_id": candidate["geoname_id"],
            "name": candidate["canonical_name"],
            "matched_name": candidate["matched_name"],
            "country": candidate["country_code"],
            "feature": candidate["feature_code"],
        }
        for candidate in candidates[:5]
    ]
    selected = ollama_json(
        "Select the candidate that best matches the museum locality. "
        "Choose only a supplied geoname_id, or null when the evidence is "
        "insufficient. Return JSON with only 'geoname_id'.\n"
        f"Locality: {locality}\nCountry: {country}\n"
        f"Candidates: {json.dumps(options)}",
        max_tokens=40,
    ).get("geoname_id")

    if selected is None:
        return None
    return next(
        (
            candidate
            for candidate in candidates[:5]
            if str(candidate["geoname_id"]) == str(selected)
        ),
        None,
    )


In [ ]:
def haversine_km(latitude, longitude, reference_latitude, reference_longitude):
    latitude, longitude, reference_latitude, reference_longitude = np.radians(
        [latitude, longitude, reference_latitude, reference_longitude]
    )
    delta_latitude = reference_latitude - latitude
    delta_longitude = reference_longitude - longitude
    value = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(latitude)
        * np.cos(reference_latitude)
        * np.sin(delta_longitude / 2) ** 2
    )
    return 6371.0088 * 2 * np.arctan2(np.sqrt(value), np.sqrt(1 - value))


def evaluate_record(row):
    ranked = vector_rank(row.label_verbatim, row.Country, candidate_pool(row))
    selected = llm_choice(row.label_verbatim, row.Country, ranked)
    distances = [
        haversine_km(
            candidate["latitude"],
            candidate["longitude"],
            row.lat_manual,
            row.long_manual,
        )
        for candidate in ranked[:5]
    ]
    selected_distance = (
        haversine_km(
            selected["latitude"],
            selected["longitude"],
            row.lat_manual,
            row.long_manual,
        )
        if selected
        else np.nan
    )
    return {
        "candidate_count": len(ranked),
        "vector_top1_geoname_id": ranked[0]["geoname_id"] if ranked else None,
        "vector_top1_distance_km": distances[0] if distances else np.nan,
        "vector_top1_within_radius": bool(
            distances and distances[0] <= row.radius_km_manual
        ),
        "vector_top5_within_radius": any(
            distance <= row.radius_km_manual for distance in distances
        ),
        "llm_geoname_id": selected["geoname_id"] if selected else None,
        "llm_distance_km": selected_distance,
        "llm_within_radius": bool(
            selected and selected_distance <= row.radius_km_manual
        ),
    }


In [ ]:
evaluations = [
    evaluate_record(row)
    for row in tqdm(benchmarks.itertuples(index=False), total=len(benchmarks))
]
results = pd.concat(
    [benchmarks.reset_index(drop=True), pd.DataFrame(evaluations)],
    axis=1,
)


In [ ]:
metrics = pd.Series(
    {
        "records": len(results),
        "retrieval_resolved": results["candidate_count"].gt(0).sum(),
        "retrieval_coverage": results["candidate_count"].gt(0).mean(),
        "vector_top1_accuracy": results["vector_top1_within_radius"].mean(),
        "vector_top1_precision": (
            results["vector_top1_within_radius"].sum()
            / results["candidate_count"].gt(0).sum()
        ),
        "vector_top5_accuracy": results["vector_top5_within_radius"].mean(),
        "vector_median_top1_distance_km": results[
            "vector_top1_distance_km"
        ].median(),
        "llm_resolved": results["llm_geoname_id"].notna().sum(),
        "llm_coverage": results["llm_geoname_id"].notna().mean(),
        "llm_top1_accuracy": results["llm_within_radius"].mean(),
        "llm_top1_precision": (
            results["llm_within_radius"].sum()
            / results["llm_geoname_id"].notna().sum()
        ),
        "llm_median_distance_km": results["llm_distance_km"].median(),
    }
)
metrics


In [ ]:
results.loc[
    ~results["llm_within_radius"],
    [
        "benchmark",
        "Barcode",
        "label_verbatim",
        "Country",
        "vector_top1_distance_km",
        "llm_distance_km",
        "candidate_count",
    ],
].head(25)


In [ ]:
connection.close()
